# USIA — build it yourselfSkeleton. Fill in the `TODO`s. `notebooks/01_pipeline_explorer.ipynb` and `src/*.py`hold a working version — use them when you're stuck, but write your own first.The point is that you can defend every number to IV.**Order:** Part A needs no downloads. Part B needs `--reference`. Part C needs 2016+2017.

## 0. Setup — given, just run it

In [ ]:
%load_ext autoreload%autoreload 2import sysfrom pathlib import PathANALYSIS = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()sys.path.insert(0, str(ANALYSIS))import config as Cimport pandas as pd, numpy as np, geopandas as gpdimport matplotlib.pyplot as pltpd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)print("streets file found:", C.STREETS_GPKG.exists())

---# Part A — Which segments can we even analyse?No downloads needed. Everything here is already on disk.

### A1. Load the study segmentsLoad the streets layer from `C.STREETS_GPKG`, layer `C.STREETS_LAYER`.Check: how many rows? What CRS? How many are treatment vs control?

In [ ]:
# TODO: read the GeoPackage into `st`st = ...# TODO: print len(st), st.crs, and the treatment_or_control counts

### A2. Fix the ID type`street_segment_id` is stored as float here (`211.0`) but as text elsewhere (`'211'`).Cast it to a consistent string, or every later merge silently matches nothing.Hint: going straight from float to `str` gives `'211.0'`. Route through `Int64` first.

In [ ]:
# TODO: add a column `sid` holding the canonical string idst["sid"] = ...# CHECK: should print 315 unique ids, and none containing a dotprint(st.sid.nunique(), st.sid.str.contains(r"\.").sum())

### A3. Decode IV's quarter codesRead `C.QTR_ATTRS_CSV`. The `QUARTER` column looks like `1819Q1`.It is a **financial** year, not a calendar year:- `1819` means FY 2018-19- Q1 = Jul-Sep 2018, Q2 = Oct-Dec 2018, Q3 = Jan-Mar 2019, Q4 = Apr-Jun 2019- Two-digit years roll at 90, so `9900` is 1999-00Write a function that turns the code into the first day of that quarter.

In [ ]:
def quarter_to_date(code: str) -> pd.Timestamp:    """Convert a financial-year quarter code to the first day of that quarter.    Parameters:        code (str): e.g. '1819Q1'    Returns:        pd.Timestamp: e.g. Timestamp('2018-07-01')    """    # TODO    ...# CHECK: all three must passassert quarter_to_date("1819Q1") == pd.Timestamp("2018-07-01")assert quarter_to_date("1213Q3") == pd.Timestamp("2013-01-01")assert quarter_to_date("9900Q2") == pd.Timestamp("1999-10-01")print("quarter decode OK")

### A4. Find each segment's intervention dateIn the quarterly file, `InterventionType` is only filled in on the quarter it changed.For each segment, find the **earliest** quarter where `InterventionType` is present andis not `'Control'`. That is the intervention date.

In [ ]:
# TODO: load the quarterly panel, add sid + a decoded date columnq = ...# TODO: build `first` — one row per segment with its earliest intervention datefirst = ...# TODO: merge onto st, then sanity-check against known Melbourne history:#   Peel St and Exhibition St should land in 2020 (the COVID pop-up lanes).#   If they land in 2021, your Q3/Q4 year offset is wrong.

### A5. Who can the sensors actually measure?The City of Melbourne archive runs **Jan 2011 to May 2020**. Sensors stopped after that.A segment is usable for before/after only if it is in the CBD, its intervention fallsinside that window, and there is enough data either side (say 180 days minimum).

In [ ]:
# TODO: add a boolean column `sensor_usable`# TODO: print how many treatment segments qualify, and list them.# You should get a small number. If you get most of the 96, re-read the date logic.

### A6. Stop and thinkAnswer these in a markdown cell before moving on — they shape the whole project:1. How many treatment segments can the sensors measure? What happens to the rest?2. Peel St and Exhibition St were treated in 2020. What can sensors still tell you   about them, and what must come from aerial imagery?3. Which single street gives you the strongest before/after, and why that one?

In [ ]:
# TODO (markdown): your answers

---# Part B — The gate: joining sensors to your streetsNeeds `python src/00_download_com_data.py --reference` to have finished.Sensor events identify a bay by `StreetMarker`. Your segments are lines. Nothingconnects them until you join bay locations to street centrelines. If this fails,nothing downstream exists.

### B1. Load the parking baysRead `data/raw/on-street-parking-bays.geojson`.It arrives in lat/lon (EPSG:4326). Reproject to `C.CRS_PROJECT` (7899) — you cannotmeasure distances in degrees. Then reduce each bay polygon to its centroid.

In [ ]:
# TODO: load, reproject, take centroidsbays = ...print(len(bays), bays.crs)

### B2. Join bays to segmentsUse `gpd.sjoin_nearest` with a `max_distance` — 25 m is a reasonable start.Think about why a distance limit matters: without one, every bay in Melbourne matches*something*.A bay could match two segments at a corner. Keep the closest.

In [ ]:
# TODO: spatial join, keep nearest match per baylookup = ...# CHECK — this is the gate. Record these numbers.print("bays matched   :", len(lookup))print("segments hit   :", lookup.sid.nunique(), "of 39 CBD segments")

### B3. Judge the result- How many of your 39 CBD segments actually have bays on them?- Does William Street have bays? (It is your flagship — if not, that is a crisis.)- Try `max_distance` at 10 m and 50 m. How much does the answer move? If it moves a  lot, your result is an artefact of an arbitrary choice and you must justify it in  the report.

In [ ]:
# TODO: bays per segment, and the sensitivity check

---# Part C — UtilisationNeeds the 2016 and 2017 archives.**The key idea:** each row is one parking *event* (a car arrived, then left).It is not a snapshot. So utilisation is never a row count.    utilisation of a bay in an hour = occupied minutes / 60A 3-hour stay contributes to three different hours. You have to spread each eventacross the hours it touches.

### C1. Peek at the raw fileRead the first ~50,000 rows out of the 2016 zip without extracting it(`zipfile.ZipFile`, then `pd.read_csv(..., nrows=50_000)`).What are the columns actually called? Which is the bay id? Do the timestamps parse?

In [ ]:
# TODO

### C2. Filter earlyThe full year is ~34 million rows. Most are irrelevant to you.Read in chunks, and inside each chunk keep only rows whose bay id is in your Part Blookup. Filter *before* parsing dates — date parsing is the slow part.

In [ ]:
# TODO: chunked read + filter. Report what fraction of rows you keep.

### C3. CleanCity of Melbourne document these problems themselves:- some durations are negative (arrival logged after departure)- some events have missing times, back-filled to midnight, which fakes very long staysDecide what to drop and write down why. IV asked for documented, defensible choices.

In [ ]:
# TODO

### C4. Events to occupied minutesFor each event, work out how many minutes it occupied in each hour it overlapped.Then group to segment x date x hour:    utilisation = total occupied minutes / (number of bays x 60)

In [ ]:
# TODO

### C5. Before vs after on William StreetIntervention: April 2017. Compare the 12 months before with the 12 months after,weekdays only, 08:00-18:00.**Report `n_bays` alongside utilisation.** If the bike lane removed bays, utilisationis a fraction over a shrinking denominator — it can rise while the actual number ofparked cars falls. That distinction is the whole policy question.

In [ ]:
# TODO

### C6. Sanity checks before you believe any of it- Do utilisation values sit between 0 and 1? Anything above 1 is a bug.- Plot an average day. Is it low overnight and high midday? If not, check your timezone.- Run the same before/after on a CBD **control** street (Lonsdale, Bourke, Russell).  If controls shift as much as William St, you are measuring noise, not the bike lane.

In [ ]:
# TODO

---## Done? Move it into `src/`Anything here that works and you will run twice belongs in `src/` with a docstring.Notebooks are for exploring; `src/` is what IV can actually reuse.